In [1]:
import json
from kafka import KafkaProducer

In [2]:
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

In [3]:
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_2019-10.csv.gz

In [ ]:
!gzip -dc green_tripdata_2019-10.csv.gz > green_tripdata_2019-10.csv

In [4]:
!wc -l green_tripdata_2019-10.csv

476387 green_tripdata_2019-10.csv


In [5]:
import pandas as pd

In [6]:
df = pd.read_csv("green_tripdata_2019-10.csv", dtype={'passenger_count': 'Int64'})
df = df.fillna(0)
#df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
#df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
df_selected = df[['lpep_pickup_datetime','lpep_dropoff_datetime','PULocationID','DOLocationID','passenger_count','trip_distance','tip_amount']]

/tmp/ipykernel_8982/1690945422.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("green_tripdata_2019-10.csv", dtype={'passenger_count': 'Int64'})


In [7]:
df_selected.dtypes

lpep_pickup_datetime      object
lpep_dropoff_datetime     object
PULocationID               int64
DOLocationID               int64
passenger_count            Int64
trip_distance            float64
tip_amount               float64
dtype: object

In [8]:
df_selected

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount
0,2019-10-01 00:26:02,2019-10-01 00:39:58,112,196,1,5.88,0.00
1,2019-10-01 00:18:11,2019-10-01 00:22:38,43,263,1,0.80,0.00
2,2019-10-01 00:09:31,2019-10-01 00:24:47,255,228,2,7.50,0.00
3,2019-10-01 00:37:40,2019-10-01 00:41:49,181,181,1,0.90,0.00
4,2019-10-01 00:08:13,2019-10-01 00:17:56,97,188,1,2.52,2.26
...,...,...,...,...,...,...,...
476381,2019-10-31 23:30:00,2019-11-01 00:00:00,65,102,0,7.04,0.00
476382,2019-10-31 23:03:00,2019-10-31 23:24:00,129,136,0,0.00,0.00
476383,2019-10-31 23:02:00,2019-10-31 23:23:00,61,222,0,3.90,0.00
476384,2019-10-31 23:42:00,2019-10-31 23:56:00,76,39,0,3.08,0.00


In [ ]:
from time import time

t0 = time()

topic_name = 'green-trips'

for index, row in df.iterrows():
    message = {
        'lpep_pickup_datetime': row['lpep_pickup_datetime'],
        'lpep_dropoff_datetime': row['lpep_dropoff_datetime'],
        'PULocationID': row['PULocationID'],
        'DOLocationID': row['DOLocationID'],
        'passenger_count': row['passenger_count'],
        'trip_distance': row['trip_distance'],
        'tip_amount': row['tip_amount']
    }
    
    producer.send(topic_name, value=message)
    print(f"Sent: {message}")

producer.flush()
producer.close()

t1 = time()

In [10]:
print(f'took {(t1 - t0):.2f} seconds')

took 101.90 seconds
